In [1]:
!git clone -b gat-v3 --single-branch https://github.com/ravil75/MetroMoscow.git
!mv MetroMoscow/{.,}* . 2>/dev/null
!rmdir MetroMoscow

Cloning into 'MetroMoscow'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 244 (delta 37), reused 67 (delta 17), pack-reused 154 (from 1)
Receiving objects: 100% (244/244), 2.29 MiB | 10.05 MiB/s, done.
Resolving deltas: 100% (106/106), done.


In [4]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [5]:
import numpy as np
import pandas as pd

from src.data_prep import load_hourly, make_pivot
from src.backtest import make_rolling_folds, summarize_results
from src.baseline_models import make_model_registry, forecast_metrics

NIGHT_HOURS = {0, 1, 2, 3, 4, 5}

hourly_src = load_hourly()
pivot = make_pivot(hourly_src, top_n=1500)

models = make_model_registry()

def day_mask(index):
    return np.array([ts.hour not in NIGHT_HOURS for ts in index])

def eval_baselines(horizon, step_hours, last_n=2, min_train_hours=96):
    folds = make_rolling_folds(
        n_hours=len(pivot),
        horizon=horizon,
        min_train_hours=min_train_hours,
        step_hours=step_hours,
    )[-last_n:]

    rows = []

    for fold in folds:
        train = pivot.iloc[:fold["train_end"]]
        test = pivot.iloc[fold["test_start"]:fold["test_end"]]
        target_index = test.index
        mask = day_mask(target_index)

        if not mask.any():
            continue

        for object_id in pivot.columns:
            series = train[object_id]
            y_true = test[object_id].values[mask]

            for model_name, predict_fn in models.items():
                y_pred = predict_fn(series, target_index, horizon, None)
                y_pred = np.asarray(y_pred)[mask]

                row = forecast_metrics(
                    y_true,
                    y_pred,
                    model_name=model_name,
                    horizon=horizon,
                    train_mode="real_only",
                    object_id=object_id,
                    fold=fold["fold"],
                )
                row.update({
                    "test_start": target_index[0],
                    "test_end": target_index[-1],
                    "train_hours": len(series),
                    "test_data": "real",
                })
                rows.append(row)

    results = pd.DataFrame(rows)
    summary = summarize_results(results)
    return results, summary

baseline_results_24, baseline_summary_24 = eval_baselines(
    horizon=24,
    step_hours=24,
    last_n=2,
)

baseline_results_1, baseline_summary_1 = eval_baselines(
    horizon=1,
    step_hours=6,
    last_n=2,
)

display(baseline_summary_24)
display(baseline_summary_1)

baseline_results_24.to_csv("eda_output/baseline_results_24h_last2_day.csv", index=False)
baseline_results_1.to_csv("eda_output/baseline_results_1h_last2_day.csv", index=False)
baseline_summary_24.to_csv("eda_output/baseline_summary_24h_last2_day.csv", index=False)
baseline_summary_1.to_csv("eda_output/baseline_summary_1h_last2_day.csv", index=False)

,horizon,train_mode,model,MAE,RMSE,MAPE,SMAPE,WAPE,n_objects,n_rows
7,24,real_only,kNN Lag,112.1257,138.7757,237.3422,50.2229,52.1143,1500,3000
5,24,real_only,Seasonal Naive,116.6336,168.1748,240.7215,42.5420,55.0582,1500,3000
2,24,real_only,Holiday Profile,122.8416,179.8876,251.4516,43.2139,56.7825,1500,3000
4,24,real_only,Same-Type Day,122.8416,179.8876,251.4516,43.2139,56.7825,1500,3000
1,24,real_only,ETS,137.9963,197.6364,303.7454,49.7525,68.1684,1500,3000
0,24,real_only,Clean Ensemble,146.0990,212.7264,319.1076,51.2395,66.5676,1500,3000
6,24,real_only,Weighted Profile,184.4697,272.1452,421.0182,57.6252,84.1330,1500,3000
3,24,real_only,Mean Profile,194.0705,288.3804,444.4102,59.0228,88.6240,1500,3000


,horizon,train_mode,model,MAE,RMSE,MAPE,SMAPE,WAPE,n_objects,n_rows
0,1,real_only,Clean Ensemble,38.9553,38.9553,36.8110,26.8987,15.4128,1500,3000
7,1,real_only,kNN Lag,48.7896,48.7896,32.5221,24.6802,18.5333,1500,3000
1,1,real_only,ETS,57.4464,57.4464,41.0296,31.3427,21.5617,1500,3000
2,1,real_only,Holiday Profile,58.0300,58.0300,43.7375,26.2159,21.7155,1500,3000
4,1,real_only,Same-Type Day,58.0300,58.0300,43.7375,26.2159,21.7155,1500,3000
5,1,real_only,Seasonal Naive,58.0300,58.0300,43.7375,26.2159,21.7155,1500,3000
6,1,real_only,Weighted Profile,268.0275,268.0275,110.4403,171.4280,91.1485,1500,3000
3,1,real_only,Mean Profile,269.4480,269.4480,109.7654,172.2842,91.5889,1500,3000
